# Resilient Plant · 06 · Financial Risk Audit Playbook (PDF)

**Final Deliverable.** Compile all 5 notebooks' outputs into the **Financial Risk Audit PDF** — the document the CFO presents to the capital allocation committee.

## Setup — auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
# Folder layout expected:
#   Module 01/
#     ├── Track 03/
#     │   ├── dataset/                   (5 CSVs: 2014..2018_Financial_Data.csv)
#     │   └── resilient_plant/
#     │       ├── data/
#     │       ├── outputs/
#     │       ├── figures/
#     │       └── (notebooks live here, or in a notebooks/ subfolder)

from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "resilient_plant":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATASET_DIR}")
print(f"Data dir     : {DATA_DIR}")
print(f"Outputs dir  : {OUTPUTS_DIR}")
print(f"Figures dir  : {FIGURES_DIR}")

# Locate the 5 yearly CSVs — accept dataset/ or data/
YEARS = [2014, 2015, 2016, 2017, 2018]
csv_paths = {}
for y in YEARS:
    name = f"{y}_Financial_Data.csv"
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[y] = folder / name
            break
assert len(csv_paths) == 5, (
    f"Expected 5 CSVs (2014..2018_Financial_Data.csv) in {DATASET_DIR} or {DATA_DIR}; "
    f"found {sorted(csv_paths.keys())}"
)
print(f"Found 5 yearly CSVs : {sorted(csv_paths.keys())}")


In [ ]:
from datetime import datetime
import pandas as pd
import numpy as np
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Image,
                                 PageBreak, Table, TableStyle)

# Load all upstream outputs
anchors = pd.read_csv(DATA_DIR / "calibration_anchors.csv")
volatility = pd.read_csv(OUTPUTS_DIR / "volatility_profile.csv", index_col=0)
profile = pd.read_csv(OUTPUTS_DIR / "shocked_vs_healthy_profile.csv", index_col=0)
empirical = pd.read_csv(OUTPUTS_DIR / "empirical_stress_frequency.csv")
sector_dist = pd.read_csv(OUTPUTS_DIR / "project_vs_sector_distribution.csv")

print("Loaded all upstream data")
print(f"  Anchors: {len(anchors)} rows")
print(f"  Volatility: {len(volatility)} rows")
print(f"  Sector distribution mapping: {len(sector_dist)} rows")


In [ ]:
pdf_path = OUTPUTS_DIR / "Financial_Risk_Audit.pdf"
doc = SimpleDocTemplate(str(pdf_path), pagesize=A4,
                        leftMargin=2*cm, rightMargin=2*cm,
                        topMargin=1.8*cm, bottomMargin=1.8*cm)

NAVY  = HexColor("#1F3864")
TEAL  = HexColor("#2E7D7D")
ORANGE= HexColor("#E07A1F")
GREEN = HexColor("#2E7D32")
RED   = HexColor("#C00000")
GREY  = HexColor("#595959")

ss = getSampleStyleSheet()
styles = {
    "title":    ParagraphStyle("title", parent=ss["Title"], fontName="Helvetica-Bold",
                                fontSize=22, textColor=NAVY, spaceAfter=6, leading=26),
    "subtitle": ParagraphStyle("subtitle", parent=ss["Normal"], fontName="Helvetica-Oblique",
                                fontSize=12, textColor=GREY, spaceAfter=18, leading=15),
    "h1":       ParagraphStyle("h1", parent=ss["Heading1"], fontName="Helvetica-Bold",
                                fontSize=16, textColor=NAVY, spaceBefore=16, spaceAfter=8, leading=20),
    "h2":       ParagraphStyle("h2", parent=ss["Heading2"], fontName="Helvetica-Bold",
                                fontSize=13, textColor=TEAL, spaceBefore=10, spaceAfter=6, leading=16),
    "body":     ParagraphStyle("body", parent=ss["BodyText"], fontName="Helvetica",
                                fontSize=10.5, leading=15, spaceAfter=6, alignment=4),
    "callout":  ParagraphStyle("callout", parent=ss["Normal"], fontName="Helvetica-Bold",
                                fontSize=11, textColor=ORANGE, leading=14, spaceAfter=8),
    "danger":   ParagraphStyle("danger", parent=ss["Normal"], fontName="Helvetica-Bold",
                                fontSize=11, textColor=RED, leading=14, spaceAfter=8),
}

# Recompute NPV/IRR for the cover (use the same Python sanity check from Notebook 03)
import numpy as np
import numpy_financial as npf
def calc_npv(wacc, op_margin, capex_y0=30e6, capex_y1=25e6, capex_y2=20e6,
             rev_y3=100e6, growth=0.05, tax=0.291,
             wc_pct=0.10, maint_pct=0.053, depr_yrs=10, g_term=0.02):
    # Corrected DCF - op_margin is post-depreciation EBIT margin.
    revs = [0, 0, 0]
    for y in range(3, 11):
        if y == 3: revs.append(rev_y3)
        else: revs.append(revs[-1] * (1 + growth))
    total_capex = capex_y0 + capex_y1 + capex_y2
    depr = total_capex / depr_yrs
    fcfs = []
    for y in range(11):
        rev = revs[y]
        ebit = rev * op_margin              # already post-depreciation
        d_addback = 0 if y < 3 else depr
        t = max(0, ebit) * tax              # tax base = EBIT
        nopat = ebit - t
        if y == 0: cap = -capex_y0
        elif y == 1: cap = -capex_y1
        elif y == 2: cap = -capex_y2
        else: cap = -rev * maint_pct
        if y < 3: dwc = 0
        elif y == 3: dwc = -rev * wc_pct
        else: dwc = -(revs[y] - revs[y-1]) * wc_pct
        fcfs.append(nopat + d_addback + cap + dwc)
    tv = fcfs[10] * (1 + g_term) / (wacc - g_term)
    fcfs[10] += tv
    npv = sum(fcfs[y] / (1 + wacc)**y for y in range(11))
    try:
        irr = npf.irr(fcfs)
    except Exception:
        irr = float("nan")
    return npv, irr

npv_base, irr_base = calc_npv(0.10, 0.079)
print(f"Base case NPV: ${npv_base:,.0f}")
print(f"Base case IRR: {irr_base:.1%}")


## Cover & Executive Summary

In [ ]:
story = []

# COVER
story.append(Spacer(1, 4*cm))
story.append(Paragraph("Financial Risk Audit", styles["title"]))
story.append(Paragraph("Resilient Plant · $75M Capital Investment", styles["title"]))
story.append(Paragraph(f"Sector-calibrated stress test for the capital allocation committee &nbsp;·&nbsp; "
                       f"{datetime.now().strftime('%B %Y')}", styles["subtitle"]))
story.append(Spacer(1, 1*cm))
story.append(Paragraph(
    "<b>Dataset:</b> Kaggle 200-Financial-Indicators · 1,344 Basic Materials company-years · 2014–2018",
    styles["body"]))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "<b>Scope:</b> Capital budgeting (TVM/NPV/IRR) · Sector ratio envelope · 2-way WACC × Margin "
    "stress test · Empirical Danger Zone calibration · Sector resilience profile",
    styles["body"]))
story.append(PageBreak())

# EXEC SUMMARY
story.append(Paragraph("Executive Summary", styles["h1"]))
story.append(Paragraph(
    "This audit evaluates the proposed $75M smelting plant against the empirical financial profile "
    "of the US-listed Basic Materials sector. Every model input (operating margin, tax rate, "
    "CapEx ratio, debt structure) is anchored to sector-median observations. Stress-test thresholds "
    "are mapped onto the empirical distribution so the committee sees not just <i>where NPV turns "
    "negative</i> but also <i>how often the sector has actually operated in those conditions</i>.",
    styles["body"]))

story.append(Paragraph("Key findings", styles["h2"]))
findings_data = [["#", "Finding", "Detail"],
    ["1", "Base case NPV is NEGATIVE",
        f"At sector-median 7.9% op margin, 10% WACC, and $100M Y3 revenue: NPV = ${npv_base/1e6:.1f}M, IRR = {irr_base:.1%}. The project does NOT clear the hurdle at sector-typical economics."],
    ["2", "Break-even requires top-quartile operations",
        f"Project NPV turns positive only above ~12% op margin — corresponding to the top quartile of the 1,200+ metals company-years in the dataset. Median operators cannot justify this CapEx."],
    ["3", "Sector calibration drives the conclusion",
        f"Every input is anchored to sector medians (op margin 7.9%, tax 29.1%, CapEx/Rev 5.3%). The negative result is what the data actually says — not a pessimistic assumption."],
    ["4", "Empirical Danger Zone identified",
        f"In the dataset, {empirical[empirical['condition']=='Op margin < 5%']['pct_of_sector'].iloc[0]:.1f}% of metals company-years had op margin below 5%. This is the empirical 'how often does it actually happen' anchor."],
    ["5", "Liquidity matters more than leverage",
        "Recovered firms had higher cash ratios at the time of shock; debt levels were similar between recoverers and non-recoverers."],
]
t = Table(findings_data, colWidths=[1*cm, 5*cm, 10*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 9.5),
    ("VALIGN", (0,0), (-1,-1), "TOP"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
]))
story.append(t)
story.append(PageBreak())


## Section 1 — Calibration anchors

In [ ]:
story.append(Paragraph("Section 1 · Sector-Calibrated Model Inputs", styles["h1"]))
story.append(Paragraph(
    "Rather than assume model inputs, we anchor every assumption to the median of the 1,344 "
    "Basic Materials company-years in the Kaggle dataset (filtered to companies with revenue > $1M). "
    "This gives the project a defensible peer benchmark.",
    styles["body"]))

# Calibration table
anchor_data = [["Input", "Anchor (sector median)", "Used in model as"]]
anchor_lookup = {
    "Operating margin (median)":     ("7.9%",  "Op margin assumption — base case"),
    "Effective tax rate (median)":   ("29.1%", "Tax rate"),
    "CapEx / Revenue (median)":      ("5.3%",  "Maintenance CapEx ratio Y3-Y10"),
    "Net margin (median)":           ("4.1%",  "Cross-check vs operating margin chain"),
    "Gross margin (median)":         ("23.9%", "Reference benchmark"),
    "Debt / Equity (median)":        ("~0.47", "Capital structure benchmark"),
    "Current ratio (median)":        ("~2.26", "Liquidity benchmark"),
}
for _, row in anchors.iterrows():
    name = row["anchor"]
    if name in anchor_lookup:
        formatted_val, model_use = anchor_lookup[name]
        anchor_data.append([name, formatted_val, model_use])

t = Table(anchor_data, colWidths=[6*cm, 4*cm, 6.5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 9.5),
    ("ALIGN", (1,1), (1,-1), "CENTER"),
    ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))

# Sector trends image
img = FIGURES_DIR / "sector_median_trends.png"
if img.exists():
    story.append(Paragraph("Sector trends 2014-2018", styles["h2"]))
    story.append(Image(str(img), width=16*cm, height=8*cm))
story.append(PageBreak())


## Section 2 — Capital budgeting result

In [ ]:
story.append(Paragraph("Section 2 · Capital Budgeting Result", styles["h1"]))
story.append(Paragraph(
    "The full DCF is in <b>Capital_Budgeting_Model.xlsx</b>. CapEx is staggered ($30M / $25M / $20M "
    "across Y0–Y2). Production runs Y3–Y10 (8-year operating life). Year-10 terminal value is "
    "computed via Gordon growth at 2% perpetuity. WACC = 10%, all operating ratios = sector median, "
    "Y3 production revenue = $100M (~1.3x total CapEx — typical capacity utilisation for a smelter).",
    styles["body"]))

# Headline KPIs
kpi_data = [
    ["Metric", "Value", "Verdict"],
    ["NPV at 10% WACC, 7.9% margin",  f"${npv_base/1e6:.2f}M",
        "ACCEPT" if npv_base > 0 else "REJECT (at sector median)"],
    ["IRR",                            f"{irr_base:.1%}" if not np.isnan(irr_base) else "n/a",
        "Above WACC" if (not np.isnan(irr_base) and irr_base > 0.10) else "Below WACC"],
    ["Total CapEx",                    "$75.0M", "—"],
    ["Y3 Revenue (full capacity)",     "$100.0M", "—"],
    ["Operating life",                 "8 years (Y3-Y10)", "—"],
]
t = Table(kpi_data, colWidths=[7*cm, 5*cm, 4.5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 11),
    ("ALIGN", (1,1), (-1,-1), "CENTER"),
    ("BACKGROUND", (0,1), (-1,1), HexColor("#E2EFDA") if npv_base > 0 else HexColor("#FCE4D6")),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("ROWBACKGROUNDS", (0,2), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))

if npv_base > 0:
    story.append(Paragraph(
        "The base-case result is positive, but only modestly. The next section shows how quickly "
        "this changes when WACC rises or margins compress.",
        styles["callout"]))
else:
    story.append(Paragraph(
        "The base-case result is NEGATIVE. This is the audit's central finding — at sector-typical "
        "economics, the project is value-destroying. The next section identifies what conditions "
        "WOULD make it value-creating (WACC × Margin combinations).",
        styles["danger"]))
story.append(PageBreak())


## Section 3 — Stress test & Danger Zone

In [ ]:
story.append(Paragraph("Section 3 · 2-Way Stress Test", styles["h1"]))
story.append(Paragraph(
    "The full 9 × 12 NPV grid lives in <b>Stress_Test_2Way.xlsx</b> with conditional formatting "
    "(red = NPV < 0). Below is a compact extract showing how NPV changes across WACC × Op Margin.",
    styles["body"]))

# Compact stress grid for the PDF — recompute a 5x5 subset spanning sector median to break-even
waccs_pdf = [0.08, 0.10, 0.12, 0.14, 0.16]
margins_pdf = [0.05, 0.079, 0.10, 0.12, 0.14]

stress_pdf = [["WACC ↓ / Op Margin →"] + [f"{m:.1%}" for m in margins_pdf]]
for w in waccs_pdf:
    row = [f"{w:.0%}"]
    for m in margins_pdf:
        v, _ = calc_npv(w, m)
        if v >= 0:
            row.append(f"+${v/1e6:.1f}M")
        else:
            row.append(f"-${abs(v)/1e6:.1f}M")
    stress_pdf.append(row)

t = Table(stress_pdf, colWidths=[4*cm] + [3*cm]*len(margins_pdf))

# Color cells based on positive/negative
ts = TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("BACKGROUND", (0,1), (0,-1), NAVY), ("TEXTCOLOR", (0,1), (0,-1), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTNAME", (0,1), (0,-1), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 10),
    ("ALIGN", (0,0), (-1,-1), "CENTER"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
])
# Conditional cell coloring
for i, w in enumerate(waccs_pdf):
    for j, m in enumerate(margins_pdf):
        v, _ = calc_npv(w, m)
        cell_color = HexColor("#C6EFCE") if v >= 0 else HexColor("#FCE4D6")
        ts.add("BACKGROUND", (j+1, i+1), (j+1, i+1), cell_color)
t.setStyle(ts)
story.append(t)
story.append(Spacer(1, 0.4*cm))

# Danger Zone definition
story.append(Paragraph("Danger Zone — empirical anchoring", styles["h2"]))

dz_data = [["Project condition", "Op margin", "% of sector at-or-below this margin"]]
for _, row in sector_dist.iterrows():
    dz_data.append([row["threshold"], f"{row['value']:.1%}",
                    f"{row['sector_pct_at_or_below']:.1f}%"])

t = Table(dz_data, colWidths=[6.5*cm, 3*cm, 6*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 10),
    ("ALIGN", (1,1), (-1,-1), "CENTER"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
]))
story.append(t)
story.append(Spacer(1, 0.4*cm))

story.append(Paragraph(
    "<b>Reading:</b> the base case (sector-median 7.9% margin) sits in the Danger Zone at every "
    "WACC level. Project becomes value-creating only above ~12% op margin — equivalent to the "
    "<b>top quartile</b> of metals-sector operations in the dataset. This is the audit's central "
    "quantitative finding.",
    styles["danger"]))
story.append(PageBreak())


## Section 4 — Sector resilience

In [ ]:
story.append(Paragraph("Section 4 · Sector Resilience Profile", styles["h1"]))
story.append(Paragraph(
    "We split the 1,344 company-years into 'Shocked' (bottom-quartile op margin) and 'Healthy' "
    "(top 75%) cohorts and compare their balance-sheet profiles. The contrast tells us "
    "<i>which ratios distinguish firms that survive shocks from those that don't</i> — and "
    "therefore what the project's balance sheet must look like to be resilient.",
    styles["body"]))

profile_data = [["Ratio", "Healthy cohort (top 75%)", "Shocked cohort (bottom 25%)", "Gap"]]
for ratio in profile.index:
    h = profile.loc[ratio, "Healthy (top 75%)"]
    s = profile.loc[ratio, "Shocked (bottom 25%)"]
    if pd.notna(h) and pd.notna(s):
        gap = h - s
        profile_data.append([ratio, f"{h:.3f}", f"{s:.3f}", f"{gap:+.3f}"])

t = Table(profile_data, colWidths=[5*cm, 4*cm, 4*cm, 2.5*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 9.5),
    ("ALIGN", (1,1), (-1,-1), "CENTER"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
]))
story.append(t)
story.append(Spacer(1, 0.3*cm))

img = FIGURES_DIR / "shocked_vs_healthy.png"
if img.exists():
    story.append(Image(str(img), width=16*cm, height=5*cm))
story.append(Spacer(1, 0.3*cm))

story.append(Paragraph(
    "<b>Implication for the project's capital structure:</b> the gap is largest in <i>liquidity</i> "
    "ratios. Shocked firms had materially lower current/quick ratios than healthy peers. "
    "The smelting plant must be financed with sufficient working-capital cushion that, at trough, "
    "the consolidated balance sheet still presents Q3+ liquidity ratios.",
    styles["callout"]))
story.append(PageBreak())


## Section 5 — Empirical envelope visualisations

In [ ]:
story.append(Paragraph("Section 5 · Sector Volatility Envelope (Deliverable 2)", styles["h1"]))
story.append(Paragraph(
    "These charts show the 25th/75th and 10th/90th percentile bands of the metals sector for "
    "each ratio, drawn from 1,200+ observations per year. The project's required performance "
    "must fit inside these empirical bands to be defensible.",
    styles["body"]))

for img_name, title in [
    ("envelope_liquidity.png",     "Liquidity envelope"),
    ("envelope_solvency.png",      "Solvency envelope"),
    ("envelope_profitability.png", "Profitability envelope"),
]:
    img_path = FIGURES_DIR / img_name
    if img_path.exists():
        story.append(Paragraph(title, styles["h2"]))
        story.append(Image(str(img_path), width=16*cm, height=5*cm))
        story.append(Spacer(1, 0.2*cm))

story.append(PageBreak())


## Section 6 — Recommendations & Methodology

In [ ]:
story.append(Paragraph("Section 6 · Recommendations", styles["h1"]))

recs = [
    ("Reject at sector-median economics",
     f"NPV at 7.9% op margin = ${npv_base/1e6:.1f}M (NEGATIVE). Project as proposed does not "
     f"clear the hurdle. Do NOT approve on the assumption that sector-typical operations will "
     f"materialise."),
    ("Approve only if operational case for top-quartile margins is credible",
     "Project becomes value-creating above ~12% op margin (top quartile of metals sector). "
     "Sponsor must articulate concrete operational levers — automation differential, scale, "
     "input cost advantage — that justify above-median performance and quantify the gap."),
    ("Set WACC × Margin governance triggers",
     "Track quarterly: if rolling 4-quarter op margin falls below 10%, project enters watch "
     "status. Below 8% (sector median) triggers operational review and possible mothballing."),
    ("Lock in fixed-rate financing",
     "The 9×12 grid shows NPV swings of ~$2-3M per 1pp WACC change. Fixed-rate debt reduces "
     "this exposure materially and compresses the Danger Zone."),
    ("Capitalise with liquidity cushion",
     "Resilience analysis shows liquidity ratios distinguish recoverers from non-recoverers more "
     "cleanly than leverage ratios. Target current ratio ≥ 2.0 and cash ratio ≥ 0.5 even in "
     "shock years — this is what the empirical data says survives metals-sector cycles."),
    ("Re-audit annually",
     "Sector medians shift. Re-pull the dataset (or successor sources) annually and re-anchor "
     "the model. Cyclical commodity sectors require periodic recalibration."),
]
rec_rows = [["#", "Action", "Rationale"]]
for i, (a, d) in enumerate(recs, 1):
    rec_rows.append([str(i), a, d])
t = Table(rec_rows, colWidths=[1*cm, 4.5*cm, 11*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), NAVY), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME", (0,0), (-1,0), "Helvetica-Bold"),
    ("FONTSIZE", (0,0), (-1,-1), 9.5),
    ("VALIGN", (0,0), (-1,-1), "TOP"),
    ("GRID", (0,0), (-1,-1), 0.3, colors.lightgrey),
    ("ROWBACKGROUNDS", (0,1), (-1,-1), [colors.white, HexColor("#F5F5F5")]),
]))
story.append(t)
story.append(Spacer(1, 0.6*cm))

# Methodology
story.append(Paragraph("Methodology &amp; Reframing Notes", styles["h1"]))
notes = [
    ("Sector calibration",
     "All operating assumptions (margin, tax, CapEx ratio) sourced from sector medians of the "
     "1,344 Basic Materials company-years in the Kaggle 200-Financial-Indicators dataset, 2014-2018. "
     "Outliers winsorised at 1st/99th percentile per ratio."),
    ("No forward forecasting",
     "We do not forecast 2019-2023 ratios from 2014-2018 data — 5 data points is too thin for "
     "horizon = 5 prediction. Instead, we use cross-sectional dispersion (1,200+ obs per year) "
     "as the empirical volatility envelope."),
    ("Reframed Task 4 (supply/demand → resilience)",
     "The original brief specified a supply/demand equilibrium analysis requiring metals price "
     "data not in the Kaggle dataset. We substitute sector-resilience analysis: which firms "
     "survived shocks and what their ratio profiles looked like. This answers the same underlying "
     "question — <i>what happens when shock hits</i> — using only the available data."),
    ("Reframed stress axis (raw materials → op margin)",
     "Raw material prices feed into operating margin. Since margin data is in the dataset and "
     "raw material prices are not, we use op margin as the X-axis of the stress test."),
    ("Empirical Danger Zone",
     "Rather than claim probabilities, we map model-driven NPV thresholds onto percentile ranks "
     "of the empirical sector distribution. This transforms 'probability of distress' into "
     "'where the project sits in the empirical sector distribution' — a question the data can answer."),
]
for title, text in notes:
    story.append(Paragraph(f"<b>{title}</b>", styles["h2"]))
    story.append(Paragraph(text, styles["body"]))

doc.build(story)
print(f"PDF saved: {pdf_path}")
print(f"File size: {pdf_path.stat().st_size / 1024:.0f} KB")


🎉 **All 6 notebooks complete!** Final outputs in `outputs/`:

- `Financial_Risk_Audit.pdf` — final committee report
- `Capital_Budgeting_Model.xlsx` — TVM/NPV/IRR DCF model
- `Stress_Test_2Way.xlsx` — 2-way data table + break-even contour
- Plus all the supporting CSVs in `outputs/`
- All charts in `figures/`